<a href="https://colab.research.google.com/github/Lil0nly2/DataScience/blob/main/2_sql_fundamentals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 2: SQL (Structured Query Language) Fundamentals & Relational Joins

Welcome to **SQL Fundamentals** notebook.

SQL is a **declarative programming language**: instead of writing *imperative* step-by-step loops (`how` to fetch data), you declare *what* data you need, and the database query optimizer plans the most efficient execution path.

---

### 🗺️ Learning Roadmap
1. **Environment Setup**: In-memory SQLite database, interactive query runner, and sample dataset
2. **Step 1: `SELECT` (Projection)**: Column projection, aliasing (`AS`), arithmetic expressions, `DISTINCT`
3. **Step 2: `WHERE` (Filtering Predicates)**: Comparison operators, Boolean logic (`AND`/`OR`/`NOT`), `IN`, `BETWEEN`, `LIKE`, and three-valued logic (`IS NULL`)
4. **Step 3: `GROUP BY` & Aggregations**: Split-apply-combine mental model, aggregate functions (`COUNT`, `SUM`, `AVG`, `MIN`, `MAX`), and `HAVING` vs `WHERE`
5. **Step 4: `ORDER BY` & SQL Clause Execution Order**: Sorting mechanisms, multi-column ordering, and the 8-stage SQL execution pipeline
6. **Step 5: Relational Joins Mental Model**: Primary/Foreign Keys, Cartesian products, and the Venn diagram breakdown
7. **Step 6: `INNER JOIN`**: The strict intersection between related tables
8. **Step 7: `LEFT (OUTER) JOIN`**: Preserving the left table and detecting unmatched records (Anti-Joins)
9. **Step 8: `RIGHT (OUTER) JOIN`**: Preserving the right table and table symmetry
10. **Step 9: `FULL OUTER JOIN`**: Complete union of records and portable ANSI SQL emulation
11. **Step 10: Join Variations**: `CROSS JOIN` and Hierarchical Self-Joins
12. **Step 11: Summary & Practice Exercises**: Hands-on challenge queries with self-checking solutions

---

## 🛠️ Step 0: Environment Setup & Sample Database

### Mental Model: In-Memory SQLite Database
To learn SQL inside a Jupyter notebook without configuring an external server (like PostgreSQL or MySQL), we use Python's built-in **`sqlite3`** module connected to an in-memory database (`:memory:`).

We also define a helper function `run_query(sql)` that executes any SQL string and renders it as a clean **Pandas DataFrame**.

```text
+------------------------+        +--------------------------+        +--------------------------+
|  SQL Query String      |  --->  |  sqlite3 In-Memory Engine |  --->  |  Pandas DataFrame        |
|  SELECT ... FROM ...   |        |  (Executes declarative)  |        |  (HTML Table in Jupyter) |
+------------------------+        +--------------------------+        +--------------------------+
```

In [15]:
import sqlite3
import pandas as pd

# 1. Create an in-memory database connection
conn = sqlite3.connect(":memory:")

# 2. Helper function to run queries and display results as a formatted table
def run_query(sql: str) -> pd.DataFrame:
    """Execute an SQL query and return results as a Pandas DataFrame."""
    return pd.read_sql_query(sql, conn)

print(f"Environment ready! SQLite version: {sqlite3.sqlite_version}")

Environment ready! SQLite version: 3.45.1


### Creating Realistic Pedagogical Tables

To properly demonstrate all join types (especially `LEFT`, `RIGHT`, and `FULL OUTER`), our dataset is deliberately designed with specific edge cases:
- `departments` table: Contains 5 departments. Note that **Research & Development** and **Legal** currently have **no employees**.
- `employees` table: Contains 8 employees. Note that **Grace Kelly** and **Henry Ford** have a `NULL` department (they haven't been assigned a department yet).
- `projects` table: Contains departmental projects with allocated budgets.

In [20]:
# Initialize sample schema and data
conn.executescript("""
DROP TABLE IF EXISTS employees;
DROP TABLE IF EXISTS departments;
DROP TABLE IF EXISTS projects;

CREATE TABLE departments (
    department_id INTEGER PRIMARY KEY,
    department_name TEXT NOT NULL,
    location TEXT NOT NULL
);

CREATE TABLE employees (
    employee_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    department_id INTEGER,
    job_title TEXT NOT NULL,
    salary REAL NOT NULL,
    hire_date TEXT NOT NULL,
    manager_id INTEGER
);

CREATE TABLE projects (
    project_id INTEGER PRIMARY KEY,
    project_name TEXT NOT NULL,
    department_id INTEGER,
    budget REAL NOT NULL
);

-- Seed departments
INSERT INTO departments (department_id, department_name, location) VALUES
(1, 'Engineering', 'New York'),
(2, 'Marketing', 'London'),
(3, 'Sales', 'Chicago'),
(4, 'Research & Development', 'San Francisco'),
(5, 'Legal', 'Austin');

-- Seed employees
INSERT INTO employees (employee_id, name, department_id, job_title, salary, hire_date, manager_id) VALUES
(101, 'Alice Smith',     1,    'Lead Engineer',      95000, '2021-03-15', NULL),
(102, 'Bob Jones',       1,    'Backend Developer',  75000, '2022-06-01', 101),
(103, 'Charlie Brown',   2,    'Marketing Director', 88000, '2020-01-10', NULL),
(104, 'Diana Prince',    3,    'Sales Lead',         72000, '2021-11-20', NULL),
(105, 'Evan Wright',     2,    'Content Strategist', 58000, '2023-02-15', 103),
(106, 'Fiona Gallagher', 3,    'Account Executive',  64000, '2022-09-01', 104),
(107, 'Grace Kelly',     NULL, 'Apprentice Intern',  42000, '2023-07-01', 101),
(108, 'Henry Ford',      NULL, 'External Advisor',   50000, '2023-08-10', NULL);

-- Seed projects
INSERT INTO projects (project_id, project_name, department_id, budget) VALUES
(501, 'Cloud Migration', 1, 150000),
(502, 'Brand Campaign',  2,  80000),
(503, 'Global Expansion', 3, 200000),
(504, 'AI Prototype',    4, 120000);
""")

print("Sample database created! Tables: employees, departments, projects")

Sample database created! Tables: employees, departments, projects


Let's preview both tables using `run_query`:

In [23]:
# Inspect departments
run_query("SELECT * FROM departments;")

,department_id,department_name,location
0,1,Engineering,New York
1,2,Marketing,London
2,3,Sales,Chicago
3,4,Research & Development,San Francisco
4,5,Legal,Austin


In [19]:
# Inspect employees
run_query("SELECT * FROM employees;")

,employee_id,name,department_id,job_title,salary,hire_date,manager_id
0,101,Alice Smith,1.0,Lead Engineer,95000.0,2021-03-15,NaN
1,102,Bob Jones,1.0,Backend Developer,75000.0,2022-06-01,101.0
2,103,Charlie Brown,2.0,Marketing Director,88000.0,2020-01-10,NaN
3,104,Diana Prince,3.0,Sales Lead,72000.0,2021-11-20,NaN
4,105,Evan Wright,2.0,Content Strategist,58000.0,2023-02-15,103.0
5,106,Fiona Gallagher,3.0,Account Executive,64000.0,2022-09-01,104.0
6,107,Grace Kelly,NaN,Apprentice Intern,42000.0,2023-07-01,101.0
7,108,Henry Ford,NaN,External Advisor,50000.0,2023-08-10,NaN


---

## 📌 Step 1: `SELECT` — The Projection Operation

### Mental Model: Vertical Slicing of Relations
In relational algebra, `SELECT` performs a **projection** ($\pi$). While a table contains many columns (attributes), `SELECT` specifies which vertical columns to include in the output dataset.

```text
Table in Storage: [ employee_id | name | department_id | job_title | salary | hire_date | manager_id ]
                                   |                                   |
SELECT name, salary:               v                                   v
Result Set:                     [ name | salary ]
```

### Core Syntax Elements:
- `SELECT *`: Retrieve all columns (useful for exploratory inspection, but avoid in production).
- `SELECT col1, col2`: Retrieve specific columns.
- `AS alias`: Rename an output column in the result set.
- Arithmetic Expressions: Create calculated columns (e.g. `salary * 12`).
- `DISTINCT`: Eliminate duplicate rows from the output.

In [29]:
# Example 1.1: Basic projection of specific columns
query = """
SELECT
    name,
    job_title,
    salary
FROM employees
"""
run_query(query)

,name,job_title,salary
0,Alice Smith,Lead Engineer,95000.0
1,Bob Jones,Backend Developer,75000.0
2,Charlie Brown,Marketing Director,88000.0
3,Diana Prince,Sales Lead,72000.0
4,Evan Wright,Content Strategist,58000.0
5,Fiona Gallagher,Account Executive,64000.0
6,Grace Kelly,Apprentice Intern,42000.0
7,Henry Ford,External Advisor,50000.0


In [42]:
# Example 1.2: Aliasing and Computed Expressions (Annual bonus and monthly take-home)
query = """
SELECT
    name AS employee_name,
    salary AS base_salary,
    salary / 12.0 AS monthly_salary,
    salary * 0.10 AS estimated_bonus
FROM employees;
"""
run_query(query)

,employee_name,base_salary,monthly_salary,estimated_bonus
0,Alice Smith,95000.0,7916.666667,9500.0
1,Bob Jones,75000.0,6250.000000,7500.0
2,Charlie Brown,88000.0,7333.333333,8800.0
3,Diana Prince,72000.0,6000.000000,7200.0
4,Evan Wright,58000.0,4833.333333,5800.0
5,Fiona Gallagher,64000.0,5333.333333,6400.0
6,Grace Kelly,42000.0,3500.000000,4200.0
7,Henry Ford,50000.0,4166.666667,5000.0


In [69]:
# Example 1.3: DISTINCT (Unique department IDs)
# First: Get unique department IDs (notice NULL is included as a distinct value)
distinct_query = """
SELECT DISTINCT department_id
FROM employees;
"""
print("--- DISTINCT Department IDs ---")
display(run_query(distinct_query))

--- DISTINCT Department IDs ---


,department_id
0,1.0
1,2.0
2,3.0
3,NaN


---

## 🔍 Step 2: `WHERE` — Filtering Rows with Predicates

### Mental Model: Horizontal Slicing of Rows
In relational algebra, the `WHERE` clause performs a **selection** ($\sigma$). It iterates row-by-row and tests a **predicate** (a Boolean condition). Only rows where the predicate evaluates to `TRUE` pass through the filter.

```text
All Rows in Table:       Predicate: salary >= 70000
Alice   (95,000)  ---->  95,000 >= 70,000 ? TRUE   ====> [KEPT]
Bob     (75,000)  ---->  75,000 >= 70,000 ? TRUE   ====> [KEPT]
Evan    (58,000)  ---->  58,000 >= 70,000 ? FALSE  ====> [DROPPED]
```

### Key Operators:
| Category | Operators | Example |
| :--- | :--- | :--- |
| **Comparison** | `=`, `!=` or `<>`, `>`, `<`, `>=`, `<=` | `salary > 60000` |
| **Logical** | `AND`, `OR`, `NOT` | `dept = 1 AND salary > 70000` |
| **Set Membership** | `IN (val1, val2, ...)` | `department_id IN (1, 2)` |
| **Range** | `BETWEEN low AND high` (inclusive) | `salary BETWEEN 50000 AND 75000` |
| **Pattern Match**| `LIKE` (`%` = any chars, `_` = 1 char) | `name LIKE 'A%'` |
| **Null Safety** | `IS NULL`, `IS NOT NULL` | `department_id IS NULL` |

### ⚠️ The Critical Gotcha: Three-Valued Logic (`NULL`)
SQL does not use two-valued logic (`TRUE` / `FALSE`). It uses **three-valued logic**: `TRUE`, `FALSE`, and `UNKNOWN`.
`NULL` represents missing or unknown information. Therefore:
- `NULL = NULL` evaluates to **`UNKNOWN`** (not `TRUE`!).
- `department_id = NULL` will **never match any rows**.
- You must **always** use `IS NULL` or `IS NOT NULL`.

In [102]:
# Example 2.1: Combining conditions with AND, OR, and parentheses
display(run_query("SELECT * FROM employees;"))

query = """
SELECT
    name,
    department_id,
    job_title,
    salary
FROM employees
WHERE (department_id = 2 OR department_id = 3)
  AND salary >= 60000;
"""
run_query(query)

,employee_id,name,department_id,job_title,salary,hire_date,manager_id
0,101,Alice Smith,1.0,Lead Engineer,95000.0,2021-03-15,NaN
1,102,Bob Jones,1.0,Backend Developer,75000.0,2022-06-01,101.0
2,103,Charlie Brown,2.0,Marketing Director,88000.0,2020-01-10,NaN
3,104,Diana Prince,3.0,Sales Lead,72000.0,2021-11-20,NaN
4,105,Evan Wright,2.0,Content Strategist,58000.0,2023-02-15,103.0
5,106,Fiona Gallagher,3.0,Account Executive,64000.0,2022-09-01,104.0
6,107,Grace Kelly,NaN,Apprentice Intern,42000.0,2023-07-01,101.0
7,108,Henry Ford,NaN,External Advisor,50000.0,2023-08-10,NaN


,name,department_id,job_title,salary
0,Charlie Brown,2,Marketing Director,88000.0
1,Diana Prince,3,Sales Lead,72000.0
2,Fiona Gallagher,3,Account Executive,64000.0


In [73]:
# Example 2.2: IN, BETWEEN, and LIKE pattern matching
query = """
SELECT
    name,
    job_title,
    salary
FROM employees
WHERE salary BETWEEN 50000 AND 80000
  AND (job_title LIKE '%Developer%' OR job_title LIKE '%Lead%');
"""
run_query(query)

,name,job_title,salary
0,Bob Jones,Backend Developer,75000.0
1,Diana Prince,Sales Lead,72000.0


In [76]:
# Example 2.3: NULL Handling (Demonstrating why '= NULL' fails vs 'IS NULL')
broken_query = """
-- Incorrect: '=' with NULL evaluates to UNKNOWN, returning 0 rows
SELECT name, department_id FROM employees WHERE department_id = NULL;
"""
print("Rows returned with '= NULL':", len(run_query(broken_query)))

display(run_query("SELECT * FROM employees;"))
correct_query = """
-- Correct: 'IS NULL' properly detects missing values
SELECT employee_id, name, job_title, department_id
FROM employees
WHERE department_id IS NULL;
"""
print("\n--- Unassigned Employees (WHERE department_id IS NULL) ---")
run_query(correct_query)

Rows returned with '= NULL': 0


,employee_id,name,department_id,job_title,salary,hire_date,manager_id
0,101,Alice Smith,1.0,Lead Engineer,95000.0,2021-03-15,NaN
1,102,Bob Jones,1.0,Backend Developer,75000.0,2022-06-01,101.0
2,103,Charlie Brown,2.0,Marketing Director,88000.0,2020-01-10,NaN
3,104,Diana Prince,3.0,Sales Lead,72000.0,2021-11-20,NaN
4,105,Evan Wright,2.0,Content Strategist,58000.0,2023-02-15,103.0
5,106,Fiona Gallagher,3.0,Account Executive,64000.0,2022-09-01,104.0
6,107,Grace Kelly,NaN,Apprentice Intern,42000.0,2023-07-01,101.0
7,108,Henry Ford,NaN,External Advisor,50000.0,2023-08-10,NaN



--- Unassigned Employees (WHERE department_id IS NULL) ---


,employee_id,name,job_title,department_id
0,107,Grace Kelly,Apprentice Intern,None
1,108,Henry Ford,External Advisor,None


---

## 📊 Step 3: `GROUP BY` & Aggregations — Summarizing Data

### Mental Model: Split-Apply-Combine
`GROUP BY` collapses multiple rows sharing identical values in specified columns into a **single summary row**.


### Standard Aggregate Functions:
- `COUNT(*)`: Total rows in the group.
- `COUNT(column)`: Non-null values in `column`.
- `SUM(column)`: Total sum of numeric values.
- `AVG(column)`: Arithmetic mean.
- `MIN(column)` / `MAX(column)`: Lowest / highest value.

### The Crucial Distinction: `WHERE` vs `HAVING`
One of the most common SQL interview questions!
- **`WHERE`**: Filters **individual rows BEFORE** they are aggregated into groups.
- **`HAVING`**: Filters **aggregated groups AFTER** grouping has taken place.
- Rule of thumb: If your filter condition contains an aggregate function (`COUNT`, `AVG`, `SUM`), it **must** go in `HAVING`, not `WHERE`.

In [81]:
# Example 3.1: Basic aggregation across the entire table (no GROUP BY)
display(run_query("SELECT * FROM employees;"))
query = """
SELECT
    COUNT(*) AS total_employees,
    AVG(salary) AS average_salary,
    MIN(salary) AS minimum_salary,
    MAX(salary) AS maximum_salary,
    SUM(salary) AS total_payroll
FROM employees;
"""
run_query(query)

,employee_id,name,department_id,job_title,salary,hire_date,manager_id
0,101,Alice Smith,1.0,Lead Engineer,95000.0,2021-03-15,NaN
1,102,Bob Jones,1.0,Backend Developer,75000.0,2022-06-01,101.0
2,103,Charlie Brown,2.0,Marketing Director,88000.0,2020-01-10,NaN
3,104,Diana Prince,3.0,Sales Lead,72000.0,2021-11-20,NaN
4,105,Evan Wright,2.0,Content Strategist,58000.0,2023-02-15,103.0
5,106,Fiona Gallagher,3.0,Account Executive,64000.0,2022-09-01,104.0
6,107,Grace Kelly,NaN,Apprentice Intern,42000.0,2023-07-01,101.0
7,108,Henry Ford,NaN,External Advisor,50000.0,2023-08-10,NaN


,total_employees,average_salary,minimum_salary,maximum_salary,total_payroll
0,8,68000.0,42000.0,95000.0,544000.0


In [86]:
# Example 3.2: Grouping by department_id with aggregates
query = """
SELECT
    department_id,
    COUNT(*) AS headcount,
    AVG(salary) AS avg_salary,
    MAX(salary) AS highest_salary
FROM employees
WHERE department_id IS NOT NULL  -- Exclude unassigned employees BEFORE grouping
GROUP BY department_id;
"""
run_query(query)

,department_id,headcount,avg_salary,highest_salary
0,1,2,85000.0,95000.0
1,2,2,73000.0,88000.0
2,3,2,68000.0,72000.0


In [96]:
# Example 3.3: WHERE vs HAVING in action
# Question: Find departments with at least 2 employees where average salary > 70,000
query = """
SELECT
    department_id,
    COUNT(*) AS headcount,
    AVG(salary) AS avg_salary
FROM employees
WHERE department_id IS NOT NULL       -- 1. Filters raw rows before grouping
GROUP BY department_id               -- 2. Creates groups per department
HAVING COUNT(*) >= 2                 -- 3. Filters aggregated groups
   AND AVG(salary) > 70000;
"""
run_query(query)

,department_id,headcount,avg_salary
0,1,2,85000.0
1,2,2,73000.0


---

## 🗂️ Step 4: `ORDER BY` & The SQL Execution Pipeline

### Mental Model: Sorting the Final Output
In relational databases, tables are **unordered mathematical sets**. Unless you explicitly specify `ORDER BY`, the database does not guarantee any row ordering.

- `ASC`: Ascending order (lowest to highest, A $\to$ Z). **Default**.
- `DESC`: Descending order (highest to lowest, Z $\to$ A).

### ⚡ SQL Execution Order

```text
Written Syntax Order:
1. SELECT                       
2. FROM                         
3. WHERE                           
4. GROUP BY                       
5. HAVING                          
6. ORDER BY                      
                                                    
```

In [97]:
# Example 4.1: Multi-column sorting with mixed directions
# Sort by department_id ascending, then by salary descending (highest earners first within dept)
display(run_query("SELECT * FROM employees;"))
query = """
SELECT
    name,
    department_id,
    job_title,
    salary
FROM employees
WHERE department_id IS NOT NULL
ORDER BY
    department_id ASC,
    salary DESC;
"""
run_query(query)

,employee_id,name,department_id,job_title,salary,hire_date,manager_id
0,101,Alice Smith,1.0,Lead Engineer,95000.0,2021-03-15,NaN
1,102,Bob Jones,1.0,Backend Developer,75000.0,2022-06-01,101.0
2,103,Charlie Brown,2.0,Marketing Director,88000.0,2020-01-10,NaN
3,104,Diana Prince,3.0,Sales Lead,72000.0,2021-11-20,NaN
4,105,Evan Wright,2.0,Content Strategist,58000.0,2023-02-15,103.0
5,106,Fiona Gallagher,3.0,Account Executive,64000.0,2022-09-01,104.0
6,107,Grace Kelly,NaN,Apprentice Intern,42000.0,2023-07-01,101.0
7,108,Henry Ford,NaN,External Advisor,50000.0,2023-08-10,NaN


,name,department_id,job_title,salary
0,Alice Smith,1,Lead Engineer,95000.0
1,Bob Jones,1,Backend Developer,75000.0
2,Charlie Brown,2,Marketing Director,88000.0
3,Evan Wright,2,Content Strategist,58000.0
4,Diana Prince,3,Sales Lead,72000.0
5,Fiona Gallagher,3,Account Executive,64000.0


In [98]:
# Example 4.2: Sorting by an alias created in SELECT (Valid because ORDER BY runs after SELECT)
query = """
SELECT
    name,
    salary,
    salary * 1.15 AS new_projected_salary
FROM employees
ORDER BY new_projected_salary DESC;
"""
run_query(query)

,name,salary,new_projected_salary
0,Alice Smith,95000.0,109250.0
1,Charlie Brown,88000.0,101200.0
2,Bob Jones,75000.0,86250.0
3,Diana Prince,72000.0,82800.0
4,Fiona Gallagher,64000.0,73600.0
5,Evan Wright,58000.0,66700.0
6,Henry Ford,50000.0,57500.0
7,Grace Kelly,42000.0,48300.0


---

## 🔗 Step 5: Relational Joins — Foundations & Mental Models

### Why Separate Tables? (Normalization)
In relational design, we avoid redundant department details (like name and location) on every single employee row. Instead, we split them into distinct tables connected by keys:
- **Primary Key (`PK`)**: Unique identifier for each row in a table (e.g., `departments.department_id`).
- **Foreign Key (`FK`)**: Reference to a primary key in another table (e.g., `employees.department_id`).

### The Visual Cheat Sheet of Join Types:

```text
   INNER JOIN                 LEFT JOIN                 RIGHT JOIN              FULL OUTER JOIN
 (Intersection)            (All Left + Match)        (All Right + Match)          (Complete Union)

   [Table A] [Table B]       [Table A] [Table B]       [Table A] [Table B]       [Table A] [Table B]
     /---\     /---\           /---\     /---\           /---\     /---\           /---\     /---\  
    |  A  ( *** ) B |         | *** ( *** ) B |         |  A  ( *** )***|         | *** ( *** )***|
     \---/     \---/           \---/     \---/           \---/     \---/           \---/     \---/  
    Only rows with             All rows of A;            All rows of B;            All rows of A
    matching keys.             unmatched B is NULL.      unmatched A is NULL.      and all rows of B.
```

### The Mathematical Origin: Cartesian Product (`CROSS JOIN`)
Every relational join is conceptually a **Cartesian Product** ($A \times B$) filtered by an `ON` condition predicate:
1. Generate every possible pair of rows between Table A ($N$ rows) and Table B ($M$ rows) $\implies N \times M$ pairs.
2. Evaluate the `ON` condition for each pair.

In [106]:
# Demonstrating the raw Cartesian product (CROSS JOIN)
# 8 employees * 5 departments = 40 combinations!
display(run_query("SELECT * FROM departments;"))
display(run_query("SELECT * FROM employees;"))

query = """
SELECT
    e.name AS employee,
    d.department_name AS department
FROM employees e
CROSS JOIN departments d
;
"""
print("Sample of Cartesian Product (CROSS JOIN):")
run_query(query)

,department_id,department_name,location
0,1,Engineering,New York
1,2,Marketing,London
2,3,Sales,Chicago
3,4,Research & Development,San Francisco
4,5,Legal,Austin


,employee_id,name,department_id,job_title,salary,hire_date,manager_id
0,101,Alice Smith,1.0,Lead Engineer,95000.0,2021-03-15,NaN
1,102,Bob Jones,1.0,Backend Developer,75000.0,2022-06-01,101.0
2,103,Charlie Brown,2.0,Marketing Director,88000.0,2020-01-10,NaN
3,104,Diana Prince,3.0,Sales Lead,72000.0,2021-11-20,NaN
4,105,Evan Wright,2.0,Content Strategist,58000.0,2023-02-15,103.0
5,106,Fiona Gallagher,3.0,Account Executive,64000.0,2022-09-01,104.0
6,107,Grace Kelly,NaN,Apprentice Intern,42000.0,2023-07-01,101.0
7,108,Henry Ford,NaN,External Advisor,50000.0,2023-08-10,NaN


Sample of Cartesian Product (CROSS JOIN):


,employee,department
0,Alice Smith,Engineering
1,Alice Smith,Marketing
2,Alice Smith,Sales
3,Alice Smith,Research & Development
4,Alice Smith,Legal
5,Bob Jones,Engineering
6,Bob Jones,Marketing
7,Bob Jones,Sales
8,Bob Jones,Research & Development
9,Bob Jones,Legal


---

## 🤝 Step 6: `INNER JOIN` — The Strict Intersection

### Mental Model: Both Sides Must Agree
`INNER JOIN` returns a row **only** when the `ON` condition evaluates to `TRUE` for both tables.

In [110]:
# Example 6.1: INNER JOIN matching employees to departments
query = """
SELECT
    e.employee_id,
    e.name AS employee_name,
    e.job_title,
    d.department_name,
    d.location
FROM employees e
INNER JOIN departments d
    ON e.department_id = d.department_id;
"""
run_query(query)

,employee_id,employee_name,job_title,department_name,location
0,101,Alice Smith,Lead Engineer,Engineering,New York
1,102,Bob Jones,Backend Developer,Engineering,New York
2,103,Charlie Brown,Marketing Director,Marketing,London
3,104,Diana Prince,Sales Lead,Sales,Chicago
4,105,Evan Wright,Content Strategist,Marketing,London
5,106,Fiona Gallagher,Account Executive,Sales,Chicago


> **Teacher's Checkpoint:** Count the output rows above. There are **6 rows**.
> - Notice that **Grace Kelly** and **Henry Ford** are missing because their `department_id` is `NULL`.
> - Notice that **Research & Development** and **Legal** are missing because no employees match them.
> If you need to keep unmatched rows, you must use an **OUTER JOIN**!

---

## 👈 Step 7: `LEFT  JOIN` — Preserving the Left Table

### Mental Model: Keep Every Row from Table A
A `LEFT JOIN` guarantees that **every single row from the left table** (Table A) appears in the result:
- When a match exists in Table B: Columns from Table B are populated normally.
- When **no match** exists: Columns from Table B are filled with **`NULL`**.


In [111]:
# Example 7.1: LEFT JOIN preserving all employees
display(run_query("SELECT * FROM departments;"))
display(run_query("SELECT * FROM employees;"))
query = """
SELECT
    e.employee_id,
    e.name AS employee_name,
    e.job_title,
    d.department_name,
    d.location
FROM employees e
LEFT JOIN departments d
    ON e.department_id = d.department_id;
"""
run_query(query)

,department_id,department_name,location
0,1,Engineering,New York
1,2,Marketing,London
2,3,Sales,Chicago
3,4,Research & Development,San Francisco
4,5,Legal,Austin


,employee_id,name,department_id,job_title,salary,hire_date,manager_id
0,101,Alice Smith,1.0,Lead Engineer,95000.0,2021-03-15,NaN
1,102,Bob Jones,1.0,Backend Developer,75000.0,2022-06-01,101.0
2,103,Charlie Brown,2.0,Marketing Director,88000.0,2020-01-10,NaN
3,104,Diana Prince,3.0,Sales Lead,72000.0,2021-11-20,NaN
4,105,Evan Wright,2.0,Content Strategist,58000.0,2023-02-15,103.0
5,106,Fiona Gallagher,3.0,Account Executive,64000.0,2022-09-01,104.0
6,107,Grace Kelly,NaN,Apprentice Intern,42000.0,2023-07-01,101.0
7,108,Henry Ford,NaN,External Advisor,50000.0,2023-08-10,NaN


,employee_id,employee_name,job_title,department_name,location
0,101,Alice Smith,Lead Engineer,Engineering,New York
1,102,Bob Jones,Backend Developer,Engineering,New York
2,103,Charlie Brown,Marketing Director,Marketing,London
3,104,Diana Prince,Sales Lead,Sales,Chicago
4,105,Evan Wright,Content Strategist,Marketing,London
5,106,Fiona Gallagher,Account Executive,Sales,Chicago
6,107,Grace Kelly,Apprentice Intern,None,None
7,108,Henry Ford,External Advisor,None,None


### 🎯 The Powerful Left Anti-Join Pattern
How do you find all employees who **do not have a department**?
Perform a `LEFT JOIN` and add `WHERE right_table.primary_key IS NULL`!

In [ ]:
# Example 7.2: Left Anti-Join (Find orphan / unassigned employees)
query = """
SELECT
    e.employee_id,
    e.name AS employee_name,
    e.job_title,
    e.salary
FROM employees e
LEFT JOIN departments d
    ON e.department_id = d.department_id
WHERE d.department_id IS NULL;
"""
print("Employees without any assigned department:")
run_query(query)

Employees without any assigned department:


,employee_id,employee_name,job_title,salary
0,107,Grace Kelly,Apprentice Intern,42000.0
1,108,Henry Ford,External Advisor,50000.0


---

## 👉 Step 8: `RIGHT  JOIN` — Preserving the Right Table

### Mental Model: Keep Every Row from Table B
A `RIGHT JOIN` is the mirror image of a `LEFT JOIN`:
- **Every single row from the right table** (Table B) is guaranteed to appear.
- If no matching employee exists, columns from Table A are filled with **`NULL`**.

In [ ]:
# Example 8.1: Native RIGHT JOIN preserving all departments
query = """
SELECT
    d.department_id,
    d.department_name,
    d.location,
    e.name AS employee_name,
    e.job_title
FROM employees e
RIGHT JOIN departments d
    ON e.department_id = d.department_id;
"""
run_query(query)

,department_id,department_name,location,employee_name,job_title
0,1,Engineering,New York,Alice Smith,Lead Engineer
1,1,Engineering,New York,Bob Jones,Backend Developer
2,2,Marketing,London,Charlie Brown,Marketing Director
3,3,Sales,Chicago,Diana Prince,Sales Lead
4,2,Marketing,London,Evan Wright,Content Strategist
5,3,Sales,Chicago,Fiona Gallagher,Account Executive
6,4,Research & Development,San Francisco,NaN,NaN
7,5,Legal,Austin,NaN,NaN


### 🎯 The Right Anti-Join Pattern: Finding "Empty" Departments
Which departments currently have **0 employees assigned**?

In [ ]:
# Example 8.2: Right Anti-Join (Find departments with 0 staff)
# Notice how e.employee_id IS NULL filters for unstaffed departments
query = """
SELECT
    d.department_id,
    d.department_name,
    d.location
FROM employees e
RIGHT JOIN departments d
    ON e.department_id = d.department_id
WHERE e.employee_id IS NULL;
"""
print("Departments with 0 assigned staff:")
run_query(query)

Departments with 0 assigned staff:


,department_id,department_name,location
0,4,Research & Development,San Francisco
1,5,Legal,Austin


---

## 🌐 Step 9: `OUTER JOIN` — The Complete Union

### Mental Model: Nobody Gets Left Behind
A `FULL OUTER JOIN` retains **all rows from both tables**:

In [116]:
# Example 9.1: Direct FULL OUTER JOIN
display(run_query("SELECT * FROM departments;"))
display(run_query("SELECT * FROM employees;"))
query = """
SELECT
    e.name AS employee_name,
    e.job_title,
    d.department_name,
    d.location,
    d.department_id
FROM employees e
FULL OUTER JOIN departments d
    ON e.department_id = d.department_id;
"""
run_query(query)

,department_id,department_name,location
0,1,Engineering,New York
1,2,Marketing,London
2,3,Sales,Chicago
3,4,Research & Development,San Francisco
4,5,Legal,Austin


,employee_id,name,department_id,job_title,salary,hire_date,manager_id
0,101,Alice Smith,1.0,Lead Engineer,95000.0,2021-03-15,NaN
1,102,Bob Jones,1.0,Backend Developer,75000.0,2022-06-01,101.0
2,103,Charlie Brown,2.0,Marketing Director,88000.0,2020-01-10,NaN
3,104,Diana Prince,3.0,Sales Lead,72000.0,2021-11-20,NaN
4,105,Evan Wright,2.0,Content Strategist,58000.0,2023-02-15,103.0
5,106,Fiona Gallagher,3.0,Account Executive,64000.0,2022-09-01,104.0
6,107,Grace Kelly,NaN,Apprentice Intern,42000.0,2023-07-01,101.0
7,108,Henry Ford,NaN,External Advisor,50000.0,2023-08-10,NaN


,employee_name,job_title,department_name,location,department_id
0,Alice Smith,Lead Engineer,Engineering,New York,1.0
1,Bob Jones,Backend Developer,Engineering,New York,1.0
2,Charlie Brown,Marketing Director,Marketing,London,2.0
3,Diana Prince,Sales Lead,Sales,Chicago,3.0
4,Evan Wright,Content Strategist,Marketing,London,2.0
5,Fiona Gallagher,Account Executive,Sales,Chicago,3.0
6,Grace Kelly,Apprentice Intern,None,None,NaN
7,Henry Ford,External Advisor,None,None,NaN
8,None,None,Research & Development,San Francisco,4.0
9,None,None,Legal,Austin,5.0


---

## 📝 Step 11: Summary Cheat Sheet & Hands-On Exercises

### Complete Join Comparison Matrix:

| Join Type | Returns Rows From | Missing Left Values Fill | Missing Right Values Fill | Typical Use Case |
| :--- | :--- | :--- | :--- | :--- |
| **`INNER JOIN`** | Only matching keys in both | Dropped | Dropped | Standard relationship lookups |
| **`LEFT JOIN`** | All left rows + matching right | Never dropped | Filled with `NULL` | Enrich primary dataset; find missing references |
| **`RIGHT JOIN`** | All right rows + matching left | Filled with `NULL` | Never dropped | Ensure all categories/departments exist in report |
| **`FULL OUTER JOIN`** | All rows from both tables | Filled with `NULL` | Filled with `NULL` | Complete reconciliation audit between two systems |
| **`CROSS JOIN`** | Cartesian product ($N \times M$) | N/A | N/A | Generating grid combinations / test matrices |

---

### 🎯 Challenge Exercises for Students

Try writing the queries below yourself in new code cells before inspecting the solutions!

#### **Exercise 1 (Filtering & Sorting):**
> Find all employees whose salary is strictly greater than 60,000 and who have a manager assigned (`manager_id IS NOT NULL`). Sort the results by salary from highest to lowest.

#### **Exercise 2 (Aggregation & HAVING):**
> For each department that has assigned employees, calculate the number of employees and the average salary. Only display departments that have **at least 2 employees**.

In [ ]:
# ====================================================================
# STUDENT WORKSPACE: Write your solution for Exercise 1 here!
# ====================================================================
exercise_1_query = """
-- Your SQL query here
"""
# run_query(exercise_1_query)

In [ ]:
# ====================================================================
# STUDENT WORKSPACE: Write your solution for Exercise 2 here!
# ====================================================================
exercise_2_query = """
-- Your SQL query here
"""
# run_query(exercise_2_query)

In [ ]:
# ====================================================================
# STUDENT WORKSPACE: Write your solution for Exercise 3 here!
# ====================================================================
exercise_3_query = """
-- Your SQL query here
"""
# run_query(exercise_3_query)

### 💡 Reference Solutions (Expand to compare with your answers)

In [ ]:
# Solution 1:
sol_1 = """
SELECT name, job_title, salary, manager_id
FROM employees
WHERE salary > 60000
  AND manager_id IS NOT NULL
ORDER BY salary DESC;
"""
print("--- Solution 1 Output ---")
display(run_query(sol_1))

# Solution 2:
sol_2 = """
SELECT
    d.department_name,
    COUNT(e.employee_id) AS total_employees,
    AVG(e.salary) AS average_salary
FROM departments d
INNER JOIN employees e ON d.department_id = e.department_id
GROUP BY d.department_id, d.department_name
HAVING COUNT(e.employee_id) >= 2;
"""
print("\n--- Solution 2 Output ---")
display(run_query(sol_2))


--- Solution 1 Output ---


,name,job_title,salary,manager_id
0,Bob Jones,Backend Developer,75000.0,101
1,Fiona Gallagher,Account Executive,64000.0,104



--- Solution 2 Output ---


,department_name,total_employees,average_salary
0,Engineering,2,85000.0
1,Marketing,2,73000.0
2,Sales,2,68000.0



--- Solution 3 Output ---


,department_name,project_name,budget
0,Engineering,Cloud Migration,150000.0
1,Marketing,Brand Campaign,80000.0
2,Sales,Global Expansion,200000.0
3,Research & Development,AI Prototype,120000.0
4,Legal,NaN,0.0
